In [1]:
import uproot

import numpy as np
import pandas as pd

import naive

In [2]:
def GhostJetPopulator(jet_input, y_min, y_max, ny, phi_min, phi_max, nphi, epsil=1e-6, version=1):
  """
  Populates a set of input jets with ghosts jets. In particular, takes jet_input set,
  and adds a series of ny*nphi many ghost jets in a uniform (y,phi) grid.

  Parameters:
      jet_input (list): List of input jets.
      y_min, y_max (float): rapidity (y) bounds of the uniform (y,phi) grid where ghost jets are populated.
      phi_min, phi_max (float): phi-bounds of the uniform (y,phi) grid where ghost jets are populated.
      ny, nphi: number of ghost jets populated along y,phi axis respectively, resulting in N = ny*nphi many ghost jets.

      epsil (float): Energy of ghost jet, by default a soft, massless jet, with E = |p|= epsil.

      version (-1 or 1): alternating between versions of ghost jet indexing.

  Returns:
      list: List of input jets with added ghost jets.

  """

  y_axis = np.linspace(y_min, y_max, ny)
  phi_axis = np.linspace(phi_min, phi_max, nphi)


  y_grid, phi_grid = np.meshgrid(y_axis, phi_axis)

  y_phi_coordinates = np.column_stack((y_grid.ravel(), phi_grid.ravel()))

  # creating soft jets, using E = |p|= epsil,
  # and obtaining p-components through y, phi with
  # pz = E*tanh(y); pT = E**2 - pz**2,
  # px = pT*cos(phi); py = pT*sin(phi)

  pz_values = epsil * np.tanh(y_grid.ravel())
  pT_values = np.sqrt(epsil**2 - pz_values**2)
  px_values = pT_values * np.cos(phi_grid.ravel())
  py_values = pT_values * np.sin(phi_grid.ravel())

  #assigns indices to ghost jets sequentially, as len(jet_input), len(jet_input)+1, len(jet_input)+2, and so on.
  #input particles in jet_input assumed to be indexed as integers, from 0, 1, 2, ..., len(jet_input)-1.

  if version == 1:
    ghost_jets = [naive.MJet(epsil, px, py, pz, [i+len(jet_input)]) for i, (px, py, pz) in enumerate(zip(px_values.ravel(), py_values.ravel(), pz_values.ravel()))]
    for gj in ghost_jets:
        print(gj)
  elif version == -1:
    ghost_jets = [naive.MJet(epsil, px, py, pz, [-(i+1)]) for i, (px, py, pz) in enumerate(zip(px_values.ravel(), py_values.ravel(), pz_values.ravel()))]
  else:
    raise ValueError("Invalid version value. Must be -1 or 1.")
  #alternate possibility: index as negative numbers [i+len(jet_input)] --> [-(i+1)] (as i starts at 0), so one can more easily distinguish ghost jets by index


  jet_input += ghost_jets

  return jet_input

In [3]:
def fj_ghost_jets(inputjet_set, y_min, y_max, n_y, phi_min, phi_max, n_phi, epsil=1e-6):
  y_axis = np.linspace(y_min, y_max, n_y)
  phi_axis = np.linspace(phi_min, phi_max, n_phi)

  y_grid, phi_grid = np.meshgrid(y_axis, phi_axis)

  y_phi_coordinates = np.column_stack((y_grid.ravel(), phi_grid.ravel()))

  pz_values = epsil * np.tanh(y_grid.ravel())
  pT_values = np.sqrt(epsil**2 - pz_values**2)
  px_values = pT_values * np.cos(phi_grid.ravel())
  py_values = pT_values * np.sin(phi_grid.ravel())

  ghost_jets = [fastjet.PseudoJet(px_values[i], py_values[i], pz_values[i], epsil) for i in range(len(pz_values))]

  inputjet_set += ghost_jets

  for i,j in enumerate(inputjet_set):
    j.set_user_index(i)



  return inputjet_set

In [4]:
injets = naive.JetReader('PythiaEventsBatchTest.root','pdEventTree')

In [5]:
#sep files for naive, fastjet ghost jets

naive_ghosts = []

fastjet_ghosts = []

ymin = min([jet.y for jet in injets])
ymax = max([jet.y for jet in injets])

In [6]:
pi = np.pi

In [7]:
for _n in range(5):

    print("_n = {}".format(_n))
    old = len(naive_ghosts)
    naive_ghosts = GhostJetPopulator(naive_ghosts, ymin, ymax, _n, 0, 2*pi, _n)
    print("")
    print(f"dN:{len(naive_ghosts) - old}")
    print("")

_n = 0

dN:0

_n = 1
{'id': [0], 'E': 1e-06, 'px': np.float64(4.011861157000708e-10), 'py': np.float64(0.0), 'pz': np.float64(-9.99999919524847e-07), 'pT': 4.011861157000708e-10, 'p': 1e-06, 'ghost_list': [], 'phi': 0.0, 'eta': -8.514232249726463, 'm': 0.0}

dN:1

_n = 2
{'id': [1], 'E': 1e-06, 'px': np.float64(4.011861157000708e-10), 'py': np.float64(0.0), 'pz': np.float64(-9.99999919524847e-07), 'pT': 4.011861157000708e-10, 'p': 1e-06, 'ghost_list': [], 'phi': 0.0, 'eta': -8.514232249726463, 'm': 0.0}
{'id': [2], 'E': 1e-06, 'px': np.float64(2.152881097689704e-10), 'py': np.float64(0.0), 'pz': np.float64(9.999999768255146e-07), 'pT': 2.152881097689704e-10, 'p': 1e-06, 'ghost_list': [], 'phi': 0.0, 'eta': 9.136680550431818, 'm': 0.0}
{'id': [3], 'E': 1e-06, 'px': np.float64(4.011861157000708e-10), 'py': np.float64(-9.826225849089029e-26), 'pz': np.float64(-9.99999919524847e-07), 'pT': 4.011861157000708e-10, 'p': 1e-06, 'ghost_list': [], 'phi': 6.283185307179586, 'eta': -8.514232249726

In [8]:
import fastjet
#fj_ghost_jets seems to work fine, but I won't bother with it right now
#may return to it some other time
# for _n in range(5):

#     print("_n = {}".format(_n))
#     old = len(fastjet_ghosts)
#     fastjet_ghosts = fj_ghost_jets(fastjet_ghosts, ymin, ymax, _n, 0, 2*pi, _n)
#     print("")
#     print(f"dN:{len(fastjet_ghosts) - old}")
#     print("")

In [9]:
#ok, so clearly something funky is going on with the fj_ghost_jets function, may need to circle back to it later.
#nvm, forgot to swap out 'naive_ghosts' with 'fastjet_ghosts'
def GhostJetPopulator_fastjet_version(jet_input, y_min, y_max, ny, phi_min, phi_max, nphi, epsil=1e-6, version=1):
  """
  Populates a set of input jets with ghosts jets. In particular, takes jet_input set,
  and adds a series of ny*nphi many ghost jets in a uniform (y,phi) grid.

  Parameters:
      jet_input (list): List of input jets.
      y_min, y_max (float): rapidity (y) bounds of the uniform (y,phi) grid where ghost jets are populated.
      phi_min, phi_max (float): phi-bounds of the uniform (y,phi) grid where ghost jets are populated.
      ny, nphi: number of ghost jets populated along y,phi axis respectively, resulting in N = ny*nphi many ghost jets.

      epsil (float): Energy of ghost jet, by default a soft, massless jet, with E = |p|= epsil.

      version (-1 or 1): alternating between versions of ghost jet indexing.

  Returns:
      list: List of input jets with added ghost jets.

  """

  y_axis = np.linspace(y_min, y_max, ny)
  phi_axis = np.linspace(phi_min, phi_max, nphi)


  y_grid, phi_grid = np.meshgrid(y_axis, phi_axis)

  y_phi_coordinates = np.column_stack((y_grid.ravel(), phi_grid.ravel()))

  # creating soft jets, using E = |p|= epsil,
  # and obtaining p-components through y, phi with
  # pz = E*tanh(y); pT = E**2 - pz**2,
  # px = pT*cos(phi); py = pT*sin(phi)

  pz_values = epsil * np.tanh(y_grid.ravel())
  pT_values = np.sqrt(epsil**2 - pz_values**2)
  px_values = pT_values * np.cos(phi_grid.ravel())
  py_values = pT_values * np.sin(phi_grid.ravel())

  #assigns indices to ghost jets sequentially, as len(jet_input), len(jet_input)+1, len(jet_input)+2, and so on.
  #input particles in jet_input assumed to be indexed as integers, from 0, 1, 2, ..., len(jet_input)-1.

  if version == 1:
    ghost_jets = [fastjet.PseudoJet(px, py, pz, epsil) for i, (px, py, pz) in enumerate(zip(px_values.ravel(), py_values.ravel(), pz_values.ravel()))]
    for gj in ghost_jets:
        print(gj)
  elif version == -1:
    ghost_jets = [fastjet.PseudoJet(px, py, pz, epsil) for i, (px, py, pz) in enumerate(zip(px_values.ravel(), py_values.ravel(), pz_values.ravel()))]
  else:
    raise ValueError("Invalid version value. Must be -1 or 1.")
  #alternate possibility: index as negative numbers [i+len(jet_input)] --> [-(i+1)] (as i starts at 0), so one can more easily distinguish ghost jets by index


  jet_input += ghost_jets

  return jet_input

  jet_input += ghost_jets

  return jet_input

#!!!! key edit: on ghost_jets lines; needed to change "fastjet.PseudoJet(px,py,pz,E)" -> "fastjet.PseudoJet(px,py,pz,epsil)"

In [10]:
for _n in range(5):

    print("_n = {}".format(_n))
    old = len(fastjet_ghosts)
    fastjet_ghosts = GhostJetPopulator_fastjet_version(fastjet_ghosts, ymin, ymax, _n, 0, 2*pi, _n)
    print("")
    print(f"dN:{len(fastjet_ghosts) - old}")
    print("")

_n = 0

dN:0

_n = 1
[0.000000, 0.000000, -0.000001, 0.000001]

dN:1

_n = 2
[0.000000, 0.000000, -0.000001, 0.000001]
[0.000000, 0.000000, 0.000001, 0.000001]
[0.000000, -0.000000, -0.000001, 0.000001]
[0.000000, -0.000000, 0.000001, 0.000001]

dN:4

_n = 3
[0.000000, 0.000000, -0.000001, 0.000001]
[0.000001, 0.000000, 0.000000, 0.000001]
[0.000000, 0.000000, 0.000001, 0.000001]
[-0.000000, 0.000000, -0.000001, 0.000001]
[-0.000001, 0.000000, 0.000000, 0.000001]
[-0.000000, 0.000000, 0.000001, 0.000001]
[0.000000, -0.000000, -0.000001, 0.000001]
[0.000001, -0.000000, 0.000000, 0.000001]
[0.000000, -0.000000, 0.000001, 0.000001]

dN:9

_n = 4
[0.000000, 0.000000, -0.000001, 0.000001]
[0.000000, 0.000000, -0.000001, 0.000001]
[0.000000, 0.000000, 0.000001, 0.000001]
[0.000000, 0.000000, 0.000001, 0.000001]
[-0.000000, 0.000000, -0.000001, 0.000001]
[-0.000000, 0.000000, -0.000001, 0.000001]
[-0.000000, 0.000000, 0.000001, 0.000001]
[-0.000000, 0.000000, 0.000001, 0.000001]
[-0.000000, -

In [11]:
jettester = fastjet.PseudoJet(1,2,3,100)

print(jettester)

[1.000000, 2.000000, 3.000000, 100.000000]


In [12]:
naive_ghosts

In [13]:
len(naive_ghosts) == len(fastjet_ghosts)

True

In [14]:
#so, the two GhostJetPopulators work the same; can be folded into same function and distinguished by "mode"?